In [2]:
import Pkg;
Pkg.activate(@__DIR__);
Pkg.status()
Pkg.instantiate()

using DelimitedFiles
using Printf

const ng_B = [1,1,1,2,2,0,0,0,1,0,1,2,3,1,1,1,1,2,2,2,2,3,1,1,1,1,2,2]
const nh_B = [0,0,1,0,0,6,2,2,1,2,1,0,0,1,1,1,1,1,0,0,0,0,1,1,1,1,1,0]

const NUM_FUNCS  = 28
const RUNS       = 25
const D          = 30
const N_SAMPLES  = 2000          # regular sampling points (C++ indices 0..1999)
const FE_STEP    = 10 * D        # 300
const MAX_FES    = 20000 * D     # 600000
const FE_VALUES  = [FE_STEP * k for k in 1:N_SAMPLES]   # 300, 600, ..., 600000

algorithms = Dict(
    "Random" => "NoMods",
    "Sobol"  => "SobolInit",
    "Halton" => "HaltonInit",
    "ExtArch" => "ExtArchive"
)

struct TrialData
    alg::String
    ev_history::Vector{Float64}     # NaN when infeasible at that sample
    lcv_history::Vector{Float64}
    final_ev::Float64
    final_lcv::Float64
    feasible::Bool
end

function compute_cv(g_vals, h_vals, ng, nh)
    cv = 0.0
    for i in 1:ng
        cv += max(0.0, g_vals[i])           # inequality: only positive = violation
    end
    for i in 1:nh
        v = abs(h_vals[i])
        if v >= 1e-4                         # equality: |h| < 1e-4 treated as 0
            cv += v
        end
    end
    return cv
end

function load_trials(folder, alg, f)
    ffile = joinpath(folder, @sprintf("RDEx_f_F%d_D30_.txt", f))
    cfile = joinpath(folder, @sprintf("RDEx_C_F%d_D30.txt", f))

    fdata = readdlm(ffile)      # 25 × 2001
    cdata = readdlm(cfile)      # 25 × (2001 × 9) = 25 × 18009

    ng = ng_B[f]
    nh = nh_B[f]
    trials = TrialData[]

    for r in 1:RUNS
        ev_hist  = Vector{Float64}(undef, N_SAMPLES)
        lcv_hist = Vector{Float64}(undef, N_SAMPLES)

        for k in 1:N_SAMPLES                    # Julia col k = C++ index k-1
            fitness = fdata[r, k]

            base = (k - 1) * 9                   # 9 values per sample in cdata
            g_vals = [cdata[r, base + i]     for i in 1:3]
            h_vals = [cdata[r, base + 3 + i] for i in 1:6]

            cv = compute_cv(g_vals, h_vals, ng, nh)
            lcv_hist[k] = cv

            if cv > 0.0
                ev_hist[k] = NaN                 # infeasible → no EV
            else
                ev_hist[k] = fitness < 1e-8 ? 0.0 : fitness
            end
        end

        fl  = lcv_hist[end]
        fe  = ev_hist[end]
        push!(trials, TrialData(alg, ev_hist, lcv_hist, fe, fl, fl == 0.0))
    end
    return trials
end

# ── Accuracy ────────────────────────────────────────────────
function accuracy_pair(t1::TrialData, t2::TrialData)
    if t1.feasible && t2.feasible
        t1.final_ev < t2.final_ev && return (1.0, 0.0)
        t1.final_ev > t2.final_ev && return (0.0, 1.0)
        return (0.5, 0.5)
    elseif !t1.feasible && !t2.feasible
        t1.final_lcv < t2.final_lcv && return (1.0, 0.0)
        t1.final_lcv > t2.final_lcv && return (0.0, 1.0)
        return (0.5, 0.5)
    else
        return t1.feasible ? (1.0, 0.0) : (0.0, 1.0)
    end
end

# ── Speed ───────────────────────────────────────────────────
# Find the first FE at which `history[i] <= threshold`
function first_fe_at(history::Vector{Float64}, threshold::Float64)
    for i in 1:length(history)
        if !isnan(history[i]) && history[i] <= threshold
            return FE_VALUES[i]
        end
    end
    return MAX_FES + 1               # never reached
end

function speed_pair(t1::TrialData, t2::TrialData)
    if t1.feasible && t2.feasible
        # threshold = worse (higher) of the two final EVs
        thr = max(t1.final_ev, t2.final_ev)
        fe1 = first_fe_at(t1.ev_history, thr)
        fe2 = first_fe_at(t2.ev_history, thr)

    elseif !t1.feasible && !t2.feasible
        # threshold = worse (higher) of the two final LCVs
        thr = max(t1.final_lcv, t2.final_lcv)
        fe1 = first_fe_at(t1.lcv_history, thr)
        fe2 = first_fe_at(t2.lcv_history, thr)

    elseif t1.feasible && !t2.feasible
        # threshold = the infeasible trial's LCV
        thr = t2.final_lcv
        fe1 = first_fe_at(t1.lcv_history, thr)
        fe2 = first_fe_at(t2.lcv_history, thr)

    else   # t1 infeasible, t2 feasible
        thr = t1.final_lcv
        fe1 = first_fe_at(t1.lcv_history, thr)
        fe2 = first_fe_at(t2.lcv_history, thr)
    end

    fe1 < fe2 && return (1.0, 0.0)
    fe1 > fe2 && return (0.0, 1.0)
    return (0.5, 0.5)
end

# ── Main scoring ────────────────────────────────────────────
println("CEC 2026 Scoring  (Accuracy + Speed)")
println("=" ^ 64)

total = Dict(a => 0.0 for a in keys(algorithms))

for f in 1:NUM_FUNCS
    all_trials = TrialData[]
    for (name, path) in algorithms
        append!(all_trials, load_trials(path, name, f))
    end

    acc   = Dict(a => 0.0 for a in keys(algorithms))
    spd   = Dict(a => 0.0 for a in keys(algorithms))

    n = length(all_trials)
    for i in 1:n, j in (i+1):n
        t1, t2 = all_trials[i], all_trials[j]

        a1, a2 = accuracy_pair(t1, t2)
        s1, s2 = speed_pair(t1, t2)

        acc[t1.alg] += a1;  acc[t2.alg] += a2
        spd[t1.alg] += s1;  spd[t2.alg] += s2
    end

    println("\nF$f:")
    for alg in sort(collect(keys(acc)))
        tot_f = acc[alg] + spd[alg]
        @printf("  %-8s  A=%7.1f  S=%7.1f  Total=%7.1f\n", alg, acc[alg], spd[alg], tot_f)
        total[alg] += tot_f
    end
end

println("\n" * "=" ^ 64)
println("Final Ranking:")
for (alg, score) in sort(collect(total), by=x->-x[2])
    @printf("  %-8s : %.1f\n", alg, score)
end

Status `~/Desktop/Evolutionary-Algos/rdex/src/Project.toml`
  [09f84164] HypothesisTests v0.11.6


  Activating project at `~/Desktop/Evolutionary-Algos/rdex/src`


CEC 2026 Scoring  (Accuracy + Speed)



F1:
  ExtArch   A= 1237.5  S=  653.0  Total= 1890.5
  Halton    A= 1237.5  S= 1815.0  Total= 3052.5
  Random    A= 1237.5  S= 1910.0  Total= 3147.5
  Sobol     A= 1237.5  S=  572.0  Total= 1809.5

F2:
  ExtArch   A= 1237.5  S=  448.0  Total= 1685.5
  Halton    A= 1237.5  S= 1746.5  Total= 2984.0
  Random    A= 1237.5  S= 1976.0  Total= 3213.5
  Sobol     A= 1237.5  S=  779.5  Total= 2017.0

F3:
  ExtArch   A=  966.0  S=  964.0  Total= 1930.0
  Halton    A= 1232.0  S= 1234.0  Total= 2466.0
  Random    A= 1691.0  S= 1691.0  Total= 3382.0
  Sobol     A= 1061.0  S= 1061.0  Total= 2122.0

F4:
  ExtArch   A= 1235.0  S= 1281.0  Total= 2516.0
  Halton    A= 1288.0  S= 1201.5  Total= 2489.5
  Random    A= 1285.5  S= 1393.5  Total= 2679.0
  Sobol     A= 1141.5  S= 1074.0  Total= 2215.5

F5:
  ExtArch   A= 1237.5  S=  868.0  Total= 2105.5
  Halton    A= 1237.5  S= 1774.5  Total= 3012.0
  Random    A= 1237.5  S= 1856.5  Total= 3094.0
  Sobol     A= 1237.5  S=  451.0  Total= 1688.5

F6:
  ExtArch 

- Random $\equiv$ Original Code
- Halton $\equiv$ Using Halton initilaziation instead of random init
- External Archive $\equiv$ Original Code + External Archive(similar to SHADE)
- Sobol $\equiv$ Using Sobol initilaziation instead of random init